In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.datasets import fetch_openml
adult = fetch_openml('adult', version=2, as_frame=True)
df = adult.frame

# Replace '?' with NaN so missing values can be handled by the imputers
df = df.replace('?', np.nan)

NumericFeatures = [
    'age',
    'fnlwgt',
    'education-num',
    'capital-gain',
    'capital-loss',
    'hours-per-week'
]

CategoricalFeatures = [
    'workclass',
    'education',
    'marital-status',
    'occupation',
    'relationship',
    'race',
    'sex',
    'native-country'
]

# Median imputation is used for numeric features because it is less affected
# by extreme values than mean imputation. StandardScaler then puts numeric
# features on a comparable scale.
#
# For categorical features, most_frequent imputation replaces missing values
# with the most common category. OneHotEncoder converts categories into
# numerical columns so that machine learning models can process them.
#
# We considered mean imputation for numeric data, but skipped it because
# income-related numeric features can contain outliers. We also considered
# constant "Missing" imputation for categorical data, but use most_frequent
# here as a simple baseline. Ordinal encoding was skipped because most
# categorical features do not have a meaningful numerical order.

NumericPipeline = Pipeline([
    ('Imputer', SimpleImputer(strategy='median')),
    ('Scaler', StandardScaler())
])

CategoricalPipeline = Pipeline([
    ('Imputer', SimpleImputer(strategy='most_frequent')),
    ('Encoder', OneHotEncoder(handle_unknown='ignore'))
])

Preprocessor = ColumnTransformer([
    ('Numeric', NumericPipeline, NumericFeatures),
    ('Categorical', CategoricalPipeline, CategoricalFeatures)
])